## Collobert & Weston embeddings implementations



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NLPEmbeddingModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim=100, kernel_size=3, use_capitalization=False):
        """
        Deep Neural Network for NLP word embeddings based on the paper architecture
        
        Args:
            vocab_size: Size of the vocabulary
            embedding_dim: Dimension of word embeddings (wsz)
            hidden_dim: Hidden dimension for TDNN layer
            kernel_size: Kernel size for TDNN convolution
            use_capitalization: Whether to use capitalization features
        """
        super(NLPEmbeddingModel, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.use_capitalization = use_capitalization
        
        # Main word embedding lookup table
        self.word_embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Capitalization embedding if needed
        if use_capitalization:
            self.cap_embedding = nn.Embedding(2, 2)  # 2 classes: capitalized or not
        
        # TDNN layer for sequence processing
        self.tdnn = nn.Conv1d(
            in_channels=embedding_dim + (2 if use_capitalization else 0),
            out_channels=hidden_dim,
            kernel_size=kernel_size,
            padding=(kernel_size - 1) // 2
        )
        
        # Additional hidden layer
        self.hidden = nn.Linear(hidden_dim, hidden_dim)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform"""
        for module in self.modules():
            if isinstance(module, (nn.Linear, nn.Conv1d, nn.Embedding)):
                nn.init.xavier_uniform_(module.weight)
                if hasattr(module, 'bias') and module.bias is not None:
                    nn.init.constant_(module.bias, 0)
    
    def forward(self, word_ids, cap_features=None):
        """
        Forward pass
        
        Args:
            word_ids: Tensor of word indices [batch_size, seq_len]
            cap_features: Tensor of capitalization features [batch_size, seq_len]
        
        Returns:
            Word embeddings [batch_size, seq_len, embedding_dim]
        """
        batch_size, seq_len = word_ids.shape
        
        # Get word embeddings
        word_embeds = self.word_embedding(word_ids)  # [batch_size, seq_len, embedding_dim]
        
        # Add capitalization features if used
        if self.use_capitalization and cap_features is not None:
            cap_embeds = self.cap_embedding(cap_features)  # [batch_size, seq_len, 2]
            combined_embeds = torch.cat([word_embeds, cap_embeds], dim=-1)
        else:
            combined_embeds = word_embeds
        
        # Apply TDNN (convolution over sequence)
        # Transpose for Conv1d: [batch_size, channels, seq_len]
        tdnn_input = combined_embeds.transpose(1, 2)
        tdnn_output = self.tdnn(tdnn_input)  # [batch_size, hidden_dim, seq_len]
        tdnn_output = F.relu(tdnn_output)
        
        # Transpose back: [batch_size, seq_len, hidden_dim]
        tdnn_output = tdnn_output.transpose(1, 2)
        
        # Apply hidden layer
        final_embeddings = self.hidden(tdnn_output)  # [batch_size, seq_len, hidden_dim]
        final_embeddings = F.relu(final_embeddings)
        
        return final_embeddings


class Vocabulary:
    """Simple vocabulary class to map words to indices"""
    
    def __init__(self, words_list):
        """
        Initialize vocabulary
        
        Args:
            words_list: List of words in vocabulary
        """
        self.word2idx = {}
        self.idx2word = {}
        self.unknown_token = "<UNK>"
        
        # Build vocabulary
        words = [self.unknown_token] + words_list
        for idx, word in enumerate(words):
            self.word2idx[word] = idx
            self.idx2word[idx] = word
        
        self.vocab_size = len(words)
    
    def encode(self, words):
        """Convert list of words to indices"""
        return [self.word2idx.get(word, self.word2idx[self.unknown_token]) for word in words]
    
    def decode(self, indices):
        """Convert indices back to words"""
        return [self.idx2word.get(idx, self.unknown_token) for idx in indices]


def preprocess_sentence(sentence, vocab, lowercase=True):
    """
    Preprocess sentence and convert to tensor format
    
    Args:
        sentence: Input sentence string
        vocab: Vocabulary object
        lowercase: Whether to convert to lowercase
    
    Returns:
        Dictionary with word_ids and capitalization features
    """
    # Tokenize
    words = sentence.strip().split()
    
    # Preprocess words
    processed_words = []
    cap_features = []
    
    for word in words:
        # Handle capitalization
        is_capitalized = word[0].isupper() if word else 0
        
        if lowercase:
            processed_word = word.lower()
        else:
            processed_word = word
        
        processed_words.append(processed_word)
        cap_features.append(1 if is_capitalized else 0)
    
    # Convert to indices
    word_ids = vocab.encode(processed_words)
    
    return {
        'word_ids': torch.tensor([word_ids], dtype=torch.long),
        'cap_features': torch.tensor([cap_features], dtype=torch.long) if cap_features else None
    }


def create_simple_vocabulary(sentences, vocab_size=30000):
    """
    Create vocabulary from list of sentences
    
    Args:
        sentences: List of sentences
        vocab_size: Maximum vocabulary size
    
    Returns:
        Vocabulary object
    """
    from collections import Counter
    
    # Count word frequencies
    word_counts = Counter()
    for sentence in sentences:
        words = sentence.lower().split()
        word_counts.update(words)
    
    # Get most common words
    most_common_words = [word for word, count in word_counts.most_common(vocab_size)]
    
    return Vocabulary(most_common_words)


def transform_sentence_to_vectors(sentence, embedding_dim=50, vocab=None, model_path=None):
    """
    Transform a sentence into word vectors using the paper's architecture
    
    Args:
        sentence: Input sentence string
        embedding_dim: Dimension of output embeddings
        vocab: Vocabulary object (will create simple one if None)
        model_path: Path to pre-trained model (uses random weights if None)
    
    Returns:
        Tensor of word embeddings [seq_len, embedding_dim]
    """
    # Create vocabulary if not provided
    if vocab is None:
        # Simple vocabulary for demonstration
        sample_words = ["the", "cat", "sat", "on", "mat", "dog", "ran", "fast"]
        vocab = Vocabulary(sample_words)
    
    # Preprocess sentence
    processed = preprocess_sentence(sentence, vocab, lowercase=True)
    
    # Initialize model
    model = NLPEmbeddingModel(
        vocab_size=vocab.vocab_size,
        embedding_dim=embedding_dim,
        hidden_dim=100,
        kernel_size=3,
        use_capitalization=True
    )
    
    # Load pre-trained weights if available
    if model_path and os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path))
        print("Loaded pre-trained model")
    else:
        print("Using randomly initialized model")
    
    # Set model to evaluation mode
    model.eval()
    
    # Get embeddings
    with torch.no_grad():
        embeddings = model(
            processed['word_ids'], 
            processed['cap_features']
        )
    
    # Return embeddings for the first (and only) sentence
    return embeddings[0]  # [seq_len, embedding_dim]


# Example usage
if __name__ == "__main__":
    # Example sentences
    sentences = [
        "The cat sat on the mat",
        "A dog ran very fast",
        "Natural language processing is amazing"
    ]
    
    # Create vocabulary
    vocab = create_simple_vocabulary(sentences, vocab_size=1000)
    
    # Transform a sentence
    sentence = "The quick brown fox jumps"
    embeddings = transform_sentence_to_vectors(
        sentence=sentence,
        embedding_dim=50,
        vocab=vocab
    )
    
    print(f"Input sentence: {sentence}")
    print(f"Embedding shape: {embeddings.shape}")
    print(f"First word embedding: {embeddings[0][:10]}...")  # Show first 10 dimensions